<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c38-header.png" alt="Nextia Learning · Search and Ranking: Find the Best Match" width="100%">

# Learning to rank

**[Learning to rank](https://learning.nextia-ai.com/courses/search/m06/learning-to-rank/)** · Search and Ranking: Find the Best Match · Module 6, lesson 3 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** build features for search candidates, train a LambdaMART model with LightGBM on judgments, and read its feature importance.

| | |
|---|---|
| **Time** | About 40 minutes |
| **Needs** | An internet connection for the setup cells (about 10 MB). No account, no key and no model download. |
| **You should know** | Cross-encoder re-ranking, from [Cross-encoder re-ranking](https://learning.nextia-ai.com/courses/search/m06/cross-encoder-re-ranking/). |
| **Tested** | Python 3.12 with NumPy 2.5.4, pandas 3.1.1 and matplotlib 3.11.2 (bm25s 0.3.16, PyStemmer 3.1.0, sentence-transformers 6.2.0, PyTorch 2.15.0, scikit-learn 1.9.1 and LightGBM 4.7.0 where the lesson needs them), in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/search/m06/learning-to-rank/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C38/M06-L03-learning-to-rank/learning-to-rank-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: numpy 2.5.4, pandas 3.1.1, lightgbm 4.7.0, and matplotlib for the pictures. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 pandas==3.1.1 matplotlib==3.11.2 lightgbm==4.7.0
import warnings; warnings.filterwarnings("ignore")
from importlib.metadata import version
for name in ['numpy', 'pandas', 'lightgbm']:
    print(name, version(name))

## Setup: get the Larkfield data

Run the next cell. It downloads the files that this lesson needs (about 11 MB) into a folder `larkfield-search`, with the course's helper `compare.py`, and checks each file's checksum. It needs an internet connection and no account, and takes about 20 seconds.

You should see `Ready:` and the number of files. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-search` (in Colab: the **Files** panel on the left), then run the cell again.

Where the data comes from: Larkfield is an invented shop. Its products, queries and relevance judgments are derived from WANDS, the Wayfair ANnotation DataSet (MIT licence). The query log with its clicks, and the prices, stock and sale flags, are **synthetic**. The [dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C38/data/dataset-card.md) says what is real and what is made.

In [ ]:
# Setup: download the files into larkfield-search/ and check them.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C38/"
FILES = {
    "data/catalogue.csv.gz": "569c1c4c41403d59492e4cb22be61a7e634b052f67b4d32d12a9f14e77c08c52",
    "data/queries.csv": "51e0286150a6162502faf6502e90b89e4984949ddcb64f1e83f7e1d3251d8634",
    "data/judgments.csv.gz": "6a0b896c64a0e1000fe9401bfc359ae7995732af458120963408d71e50333ec6",
    "data/recorded/rerank.csv.gz": "38971117046aefd883c159ed9a7b196a0ca88f54757a8dc35df956d39222da88",
    "data/log/impressions.csv.gz": "588cacc6c4ab6ac226478013644bf64871c69c32650fefd0219df431a38d80d0",
    "compare.py": "f09c610e8e9026e0c355f693cb56cd3e178e7217c0e001a744280fb82ff867d2",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-search").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if not (path.exists() and sha256(path) == expected):
        local = HERE / name  # a copy next to the notebook, if there is one
        if local.exists():
            shutil.copy(local, path)
        else:
            download(LABS + name, path)
        if sha256(path) != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-search and run again.")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
print(f"Ready: {len(FILES)} files in larkfield-search.")

## Setup: helpers

The next cell loads NumPy as `np`, pandas as `pd`, matplotlib, and the course helper as `cp` (NDCG, recall, MRR, intervals and the comparison table). It defines a few small helpers:

- `catalogue()`, `queries()` and `judgments()` read the data: one row per product, one row per query, and `{query_id: {product_id: grade}}` with grade 2 exact, 1 partial, 0 irrelevant.
- `show(ids)` prints products with their title and class.

It also defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: the data, and checks that do not show the answer.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import compare as cp
warnings.filterwarnings("ignore")

def catalogue():
    """One row per product."""
    df = pd.read_csv("data/catalogue.csv.gz", keep_default_na=False, dtype={"title_fr": str})
    for col in ("on_sale", "in_stock"):
        df[col] = df[col].astype(str).eq("True")
    return df

def queries():
    """One row per query with judgments (train, validation and test)."""
    return pd.read_csv("data/queries.csv", keep_default_na=False)

def judgments():
    """{query_id: {product_id: grade}}: 2 exact, 1 partial, 0 irrelevant."""
    j = pd.read_csv("data/judgments.csv.gz")
    out = {}
    for q, p, g in zip(j["query_id"], j["product_id"], j["grade"]):
        out.setdefault(q, {})[p] = int(g)
    return out

def show(ids, judged=None, n=10):
    """Print products: rank, ID, title, class (and the grade when judged is given)."""
    global _CAT
    if "_CAT" not in globals():
        _CAT = catalogue().set_index("product_id")
    cat = _CAT
    names = {2: "exact", 1: "partial", 0: "irrelevant"}
    for i, p in enumerate(list(ids)[:n], 1):
        g = "" if judged is None else f"  [{names.get(judged.get(p), 'not judged')}]"
        print(f"{i:2}. {p}  {cat.loc[p, 'title'][:55]:<55}  {cat.loc[p, 'product_class'][:22]}{g}")

def fingerprint(value):
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Learn the order from pairs

A **learning-to-rank** model learns how to order the candidates of a query from examples: queries with judged products. **LambdaMART** looks at pairs. For each pair where a better product sits below a worse one, it asks: how much would NDCG gain if these two swapped? It pushes hardest on the pairs with the biggest gain. Run the next cell: three candidates, two wrong pairs.

In [ ]:
# Three candidates for one query: the model's score now, and the judges' grade.
toy = pd.DataFrame({"product": ["A", "B", "C"], "score": [2.1, 1.5, 0.3], "grade": [0, 2, 1]})
toy = toy.sort_values("score", ascending=False).reset_index(drop=True)
toy["rank"] = toy.index + 1
print(toy.to_string(index=False))

def dcg_of(grades):
    return sum((2 ** g - 1) / np.log2(r + 2) for r, g in enumerate(grades))

ideal = dcg_of(sorted(toy["grade"], reverse=True))
now = dcg_of(toy["grade"]) / ideal
print(f"\nNDCG now: {now:.3f}")
rows = []
for i in range(3):
    for k in range(i + 1, 3):
        if toy.loc[k, "grade"] > toy.loc[i, "grade"]:          # a better product sits lower: a wrong pair
            g = list(toy["grade"]); g[i], g[k] = g[k], g[i]
            rows.append({"pair": f"{toy.loc[k, 'product']} above {toy.loc[i, 'product']}",
                         "NDCG if swapped": round(dcg_of(g) / ideal, 3),
                         "gain from the swap": round(dcg_of(g) / ideal - now, 3)})
pd.DataFrame(rows)

> **Read the table.** NDCG now is 0.659. Product B (exact) below A (irrelevant) is the expensive mistake: B belongs at the top, where the discount is smallest. LambdaMART gives that pair the largest push.

## 2. Features for every candidate

The model cannot read text. It reads **features**: numbers that describe a query-product pair. The next cell builds 12 of them for the top 100 hybrid candidates of the validation and test queries: the hybrid rank, the cross-encoder's recorded score, price, stock, sale, rating, popularity in the **synthetic** log, age, and two facts about the query. The project uses 19 features, with BM25 per field; see the lesson.

In [ ]:
q = queries()
j = judgments()

In [ ]:
cat = catalogue().set_index("product_id")
imp = pd.read_csv("data/log/impressions.csv.gz")      # synthetic log: one row per result shown
pop = imp.groupby("product_id")[["clicked", "added_to_cart"]].sum()
rr = pd.read_csv("data/recorded/rerank.csv.gz")
qi = q.set_index("query_id")

f = rr.join(cat[["price", "in_stock", "on_sale", "average_rating", "rating_count", "added"]], on="product_id")
f = f.join(pop, on="product_id")
f["ce_score"] = f["score"]                                  # the cross-encoder, recorded
f["price_log"] = np.log1p(f["price"])
f["in_stock"] = f["in_stock"].astype(float)
f["on_sale"] = f["on_sale"].astype(float)
f["rating"] = pd.to_numeric(f["average_rating"], errors="coerce").fillna(0)
f["rating_count_log"] = np.log1p(f["rating_count"])
f["clicks_log"] = np.log1p(f["clicked"].fillna(0))
f["carts_log"] = np.log1p(f["added_to_cart"].fillna(0))
f["age_days"] = (pd.Timestamp("2025-10-26") - pd.to_datetime(f["added"])).dt.days.clip(lower=0)
f["query_words"] = f["query_id"].map(qi["query"].str.split().str.len())
f["french"] = (f["query_id"].map(qi["language"]) == "fr").astype(float)
f["label"] = [j.get(a, {}).get(b, 0) for a, b in zip(f["query_id"], f["product_id"])]
f["split"] = f["query_id"].map(qi["split"])
f = f[f["query_id"].map(qi["source"]) != "code"].sort_values(["query_id", "hybrid_rank"])   # the code rule answers codes

FEATS = ["hybrid_rank", "ce_score", "price_log", "in_stock", "on_sale", "rating", "rating_count_log",
         "clicks_log", "carts_log", "age_days", "query_words", "french"]
print(len(FEATS), "features;", f.groupby("split")["query_id"].nunique().to_dict(), "queries")
f[["query_id", "product_id", "label"] + FEATS].head(3)

## 3. Train LambdaMART with LightGBM

The recorded cross-encoder scores exist only for the validation and test queries. So this notebook trains on the validation queries and measures on the test queries. (The project trains on the 260 training queries and keeps validation for choices.)

> **Predict.** Will the model beat the cross-encoder alone on the test queries?

In [ ]:
import lightgbm as lgb

train = f[f["split"] == "validation"]
test = f[f["split"] == "test"]
params = {"objective": "lambdarank", "learning_rate": 0.05, "num_leaves": 15, "min_data_in_leaf": 20,
          "label_gain": [0, 1, 3], "seed": 0, "num_threads": 1, "verbose": -1}
groups = train.groupby("query_id", sort=False).size().to_numpy()      # candidates per query, in order
model = lgb.train(params, lgb.Dataset(train[FEATS], train["label"], group=groups), num_boost_round=200)

def ndcg_of(df, column):
    """Mean NDCG@10 when each query's candidates are sorted by `column` (highest first)."""
    run = {qid: list(g.sort_values([column, "product_id"], ascending=[False, True])["product_id"])
           for qid, g in df.groupby("query_id")}
    return cp.per_query(run, j)["ndcg@10"].mean()

test = test.assign(hybrid=-test["hybrid_rank"], ltr=model.predict(test[FEATS]))
print("test, hybrid order:      ", round(ndcg_of(test, "hybrid"), 3))
print("test, cross-encoder only:", round(ndcg_of(test, "ce_score"), 3))

## 4. Your turn: measure the model

In [ ]:
ltr_ndcg = ...   # mean NDCG@10 on the test queries when the candidates are sorted by the column "ltr"
print(ltr_ndcg)

In [ ]:
expect('ltr_ndcg', ltr_ndcg, 0.608, tolerance=0.01)

> **Check.** About 0.608, against 0.587 for the hybrid order and 0.598 for the cross-encoder alone (test queries without the codes). The model combines the cross-encoder's reading with the hybrid rank and the French flag. Your number can differ in the third decimal on another computer.

In [ ]:
gain = pd.Series(model.feature_importance("gain"), index=FEATS)
share = (gain / gain.sum()).sort_values()
share.plot.barh(figsize=(6, 3.5), title="Feature importance (share of the total gain)")
plt.tight_layout(); plt.show()
print(share.sort_values(ascending=False).round(3).to_string())

In [ ]:
top_feature = ...   # the name of the feature with the largest share (a string)
print(top_feature)

In [ ]:
expect_hash('top_feature', top_feature, '2ac0c63224a4072d')

> **Check.** The largest share (about 33%) goes to the hybrid rank: stage one's own order is a strong signal. Importance shows what the model uses, not what is true: a feature can be important because it repeats another one.

## 5. A number that is too good

Measure the model on the queries it learned from.

In [ ]:
# The same model on the queries it learned from: too good to be true.
seen = train.assign(hybrid=-train["hybrid_rank"], ltr=model.predict(train[FEATS]))
print("validation (training queries here):", round(ndcg_of(seen, "ltr"), 3), " hybrid order:", round(ndcg_of(seen, "hybrid"), 3))

> **Check.** About 0.789 on its own training queries, against 0.608 on new ones. A model always looks better on the queries it learned from. Measure on queries it never saw, as in [Train, validation and test sets](https://learning.nextia-ai.com/courses/ml/m03/train-validation-and-test-sets/) of the machine learning course. In the project: `search ltr train`, then `search eval --run ltr`.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Learn from clicks](https://learning.nextia-ai.com/courses/search/m06/learn-from-clicks/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/search/m06/learning-to-rank/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The products, queries and judgments come from WANDS (Wayfair, MIT licence). The query log, prices, stock and sale flags are synthetic: our code made them. No real customer is in this data.